# GRU Single-User Training (Colab)
Runs the GRU-based CTC model on subject #89335547. Keep runtime on GPU.


## 0. Runtime check

In [ ]:
import torch, platform
print('python', platform.python_version())
print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device', torch.cuda.get_device_name(0))


## 1. Install deps
Assumes this notebook sits in the repo root.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .


## 2. Get data (single subject #89335547)
Extract only the needed subject from the public tarball. Skip if you already have `/content/data`.


In [ ]:
import os, pathlib, subprocess
DATA_ROOT = pathlib.Path('/content/emg2qwerty-data-2021-08')
if DATA_ROOT.exists():
    print('Data already present at', DATA_ROOT)
else:
    os.makedirs('/content/tmp_dl', exist_ok=True)
    tar_path = '/content/tmp_dl/emg2qwerty-data-2021-08.tar.gz'
    url = 'https://fb-ctrl-oss.s3.amazonaws.com/emg2qwerty/emg2qwerty-data-2021-08.tar.gz'
    print('Downloading (may take a while)...')
    subprocess.run(['bash','-lc', f"curl -L {url} -o {tar_path}"], check=True)
    extract_cmd = f"tar -xzf {tar_path} -C /content --wildcards 'emg2qwerty-data-2021-08/metadata.csv' 'emg2qwerty-data-2021-08/89335547_*.hdf5'"
    subprocess.run(['bash','-lc', extract_cmd], check=True)
    if not DATA_ROOT.exists():
        pathlib.Path('/content/emg2qwerty-data-2021-08').rename(DATA_ROOT)
    print('Extracted to', DATA_ROOT)
if not pathlib.Path('data').exists():
    pathlib.Path('data').symlink_to(DATA_ROOT)
!ls -lh data | head


## 3. Train GRU model (single user splits)
Adjust epochs/batch size as needed.

In [ ]:
!python -m emg2qwerty.train   model=gru_ctc   user=single_user   dataset.root=/content/data   trainer.accelerator=gpu trainer.devices=1   trainer.max_epochs=40   batch_size=32


## 4. Resume / short smoke test
Run a quick 1-epoch sanity check instead of the full training.

In [ ]:
!python -m emg2qwerty.train   model=gru_ctc   user=single_user   dataset.root=/content/data   trainer.accelerator=gpu trainer.devices=1   trainer.max_epochs=1   batch_size=8
